# Integrated Project Part 3 — Validating our data

Complete submission notebook. Run cells top to bottom.

## 0. Install dependencies (run once)

In [ ]:
# Run this cell first if you are missing any packages
# %pip install sqlalchemy pandas scipy pyarrow pytest

## 1. Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import logging
from scipy.stats import ttest_ind

import data_ingestion
from field_data_processor   import FieldDataProcessor
from weather_data_processor import WeatherDataProcessor

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

## 2. Central configuration

One dictionary holds every path, query, and pattern. Change settings here — not inside the module files.

In [ ]:
config_params = {
    # SQLAlchemy connection string for the SQLite database
    "db_path": "sqlite:///Maji_Ndogo_farm_survey_small.db",

    # Joins all four survey tables into one DataFrame
    "sql_query": """
        SELECT *
        FROM geographic_features
        LEFT JOIN weather_features         USING (Field_ID)
        LEFT JOIN soil_and_crop_features   USING (Field_ID)
        LEFT JOIN farm_management_features USING (Field_ID)
    """,

    # The raw join has Annual_yield and Crop_type swapped — this fixes it
    "columns_to_rename": {
        "Annual_yield": "Crop_type",
        "Crop_type":    "Annual_yield"
    },

    # Known misspellings in the Crop_type column
    "values_to_rename": {
        "cassaval": "cassava",
        "wheatn":   "wheat",
        "teaa":     "tea"
    },

    # Raw IoT sensor messages CSV
    "weather_csv_path": (
        "https://raw.githubusercontent.com/Explore-AI/PublicData/master/"
        "Maji_Ndogo/Weather_station_data.csv"
    ),

    # Maps each Field_ID to its nearest weather station
    "weather_mapping_csv": (
        "https://raw.githubusercontent.com/Explore-AI/PublicData/master/"
        "Maji_Ndogo/Weather_data_field_mapping.csv"
    ),

    # Regex patterns to extract values from free-text sensor messages
    "regex_patterns": {
        "Rainfall":        r"(\d+(\.\d+)?)\s?mm",
        "Temperature":     r"(\d+(\.\d+)?)\s?C",
        "Pollution_level": r"=\s*(-?\d+(\.\d+)?)|Pollution at\s*(-?\d+(\.\d+)?)"
    },
}

In [ ]:
# Run this cell to find the correct URL
import urllib.request

# Try the correct URL
url = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv"

try:
    urllib.request.urlretrieve(url, "test_mapping.csv")
    print("✅ URL works!")
except Exception as e:
    print(f"❌ Failed: {e}")

## 3. Run the field data pipeline

One call to `.process()` connects to the database, cleans the data, and merges the weather station mapping.

In [ ]:
field_processor = FieldDataProcessor(config_params)
field_processor.process()
field_df = field_processor.df

# Rename Ave_temps to Temperature so both DataFrames share the same column name
field_df.rename(columns={'Ave_temps': 'Temperature'}, inplace=True)

print(field_df.shape)          # Expected: (5654, 19)
field_df['Weather_station'].unique()   # Expected: array([4, 0, 1, 2, 3])

## 4. Run the weather data pipeline

Downloads the raw sensor messages and extracts Temperature, Rainfall, and Pollution_level from the free-text Message column.

In [ ]:
weather_processor = WeatherDataProcessor(config_params)
weather_processor.process()
weather_df = weather_processor.weather_df

print(weather_df.shape)
weather_df['Measurement'].unique()   # Expected: ['Temperature', 'Pollution_level', 'Rainfall']

## 5. Automated data validation (pytest)

Saves temporary CSVs, runs 7 automated checks, then deletes the CSVs.

In [ ]:
import os

# Save DataFrames to CSV so pytest can read them
weather_df.to_csv('sampled_weather_df.csv', index=False)
field_df.to_csv('sampled_field_df.csv', index=False)

# Run the test suite
!pytest validate_data.py -v

# Clean up temporary files
for f in ['sampled_weather_df.csv', 'sampled_field_df.csv']:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")

## 6. Hypothesis testing

We use a **two-sample Welch's t-test** instead of a simple percentage tolerance because it accounts for the *spread* (variance) of each dataset — not just the means.

**Null hypothesis H₀:** There is no significant difference between the field data means and the weather station means. (μ_field = μ_weather)

**Alternative hypothesis Hₐ:** There is a significant difference. (μ_field ≠ μ_weather)

- If **p ≤ 0.05** → reject H₀ (significant difference found — data may not reflect reality)  
- If **p > 0.05** → fail to reject H₀ (no evidence of a difference — data looks good ✅)


### filter_field_data
Returns all field measurements for a given station and measurement type.

In [ ]:
def filter_field_data(df, station_id, measurement):
    """
    Filter field_df to one weather station and return a single measurement column.

    Parameters
    ----------
    df          : pd.DataFrame  — the cleaned field DataFrame
    station_id  : int           — weather station ID (0 to 4)
    measurement : str           — e.g. 'Temperature', 'Rainfall', 'Pollution_level'

    Returns
    -------
    pd.Series — the measurement values for all fields at that station
    """
    return df[df['Weather_station'] == station_id][measurement]


# --- Test ---
station_id  = 0
measurement = 'Temperature'
field_values = filter_field_data(field_df, station_id, measurement)
print(f"Shape: {field_values.shape}, First value: {field_values.iloc[0]}")
# Expected: Shape: (1375,), First value: 13.35

### filter_weather_data
Returns all sensor readings for a given station and measurement type.

In [ ]:
def filter_weather_data(df, station_id, measurement):
    """
    Filter weather_df to one weather station and one measurement type.

    Parameters
    ----------
    df          : pd.DataFrame  — the parsed weather DataFrame
    station_id  : int           — weather station ID (0 to 4)
    measurement : str           — e.g. 'Temperature', 'Rainfall', 'Pollution_level'

    Returns
    -------
    pd.Series — the Value column for the matching rows
    """
    mask = (
        (df['Weather_station_ID'] == station_id) &
        (df['Measurement']        == measurement)
    )
    return df[mask]['Value']


# --- Test ---
weather_values = filter_weather_data(weather_df, station_id, measurement)
print(f"Shape: {weather_values.shape}, First value: {weather_values.iloc[0]}")
# Expected: Shape: (100,), First value: 12.82

### run_ttest
Runs a two-sample Welch's t-test and returns the t-statistic and p-value.

In [ ]:
def run_ttest(Column_A, Column_B):
    """
    Run a two-sample Welch's t-test between two data Series.

    Uses equal_var=False (Welch's t-test) because the two samples have
    very different sizes (~1375 field readings vs ~100 weather readings).

    Parameters
    ----------
    Column_A : pd.Series — first sample  (e.g. field temperature values)
    Column_B : pd.Series — second sample (e.g. weather station temperature values)

    Returns
    -------
    tuple : (t_statistic, p_value)
    """
    t_stat, p_val = ttest_ind(Column_A, Column_B, equal_var=False, alternative='two-sided')
    return t_stat, p_val


# --- Test ---
t_stat, p_val = run_ttest(field_values, weather_values)
print(f"T-stat: {t_stat:.5f}, p-value: {p_val:.5f}")
# Expected: T-stat: -0.11632, p-value: 0.90761

### print_ttest_results
Interprets the p-value and prints a plain-English conclusion.

In [ ]:
def print_ttest_results(station_id, measurement, p_val, alpha):
    """
    Print whether the null hypothesis is rejected or not.

    Parameters
    ----------
    station_id  : int   — the weather station being tested
    measurement : str   — the measurement being compared
    p_val       : float — p-value from the t-test
    alpha       : float — significance level (typically 0.05)
    """
    if p_val <= alpha:
        print(
            f"  Significant difference in {measurement} detected at Station "
            f"{station_id}, (P-Value: {p_val:.5f} < {alpha}). "
            f"Null hypothesis rejected."
        )
    else:
        print(
            f"  No significant difference in {measurement} detected at Station "
            f"{station_id}, (P-Value: {p_val:.5f} > {alpha}). "
            f"Null hypothesis not rejected."
        )


# --- Test ---
alpha = 0.05
print_ttest_results(station_id, measurement, p_val, alpha)
# Expected: No significant difference in Temperature detected at Station 0,
#           (P-Value: 0.90761 > 0.05). Null hypothesis not rejected.

### hypothesis_results
Loops over every station and measurement, runs the t-test, and prints the result.

In [ ]:
def hypothesis_results(field_df, weather_df, list_measurements_to_compare, alpha=0.05):
    """
    Run t-tests for every combination of weather station and measurement type.

    Parameters
    ----------
    field_df                     : pd.DataFrame — cleaned field survey data
    weather_df                   : pd.DataFrame — parsed weather station data
    list_measurements_to_compare : list[str]    — measurement column names to test
    alpha                        : float        — significance level (default 0.05)
    """
    station_ids = sorted(field_df['Weather_station'].dropna().unique())

    for station_id in station_ids:
        for measurement in list_measurements_to_compare:
            field_values   = filter_field_data(field_df, station_id, measurement)
            weather_values = filter_weather_data(weather_df, station_id, measurement)
            t_stat, p_val  = run_ttest(field_values, weather_values)
            print_ttest_results(station_id, measurement, p_val, alpha)

## 7. Final results

Run the hypothesis test across all 5 stations and 3 measurements (15 tests total).

In [ ]:
measurements_to_compare = ['Temperature', 'Rainfall', 'Pollution_level']
alpha = 0.05

hypothesis_results(field_df, weather_df, measurements_to_compare, alpha)

## 8. Conclusion

For all 15 tests (5 stations × 3 measurements) the **p-value > 0.05**, so we **fail to reject the null hypothesis** every time.

This means: there is no statistically significant difference between the field survey data and the independent weather station readings.

**In plain English:** our field data accurately reflects the real weather conditions in Maji Ndogo. We can trust it and move on to Machine Learning with confidence.
